# Job Listing Prioritization for Career Changers: ML_Rank (the priority queue)

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

Builds the **listing model** and the **priority queue**. The listing model predicts how long each listing stays open (chance still open, apply-by date) from every feature that `EDA_SDA_4`'s screening rule kept, text included (2a reads them from `modeling_ready.parquet`, so this notebook holds no list of them), plus the `TimeSeries` timing features. The priority queue then **combines** it with `ML_Fit`'s fit scores, career-change accessibility and pay into one score per listing, for a given career changer. Listings a career changer can't get are already hidden (never removed). Role discovery and the fit model are in `ML_Fit`.

**There's no single "priority" label to train on,** so the two models answer separate questions (will it stay open? does it fit this person?) and a formula with evidence-based, tunable weights combines them. Every score shows its parts.

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the full file, how does tech compare with every field, and what's in the scope? (the scope cut, applied first) | Scope table and its descriptions, occupation summary, company activity |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? Additional cleaning, non-text checks by role bucket, ghost listings | Scope table, non-text features |
| EDA_SDA_3 | What do the descriptions say? Text features, text rules as labels and opt-in filters (keyword rules, not validated), skills mapped to O*NET (no matching models) | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

### Clarifying questions
- How well can we predict whether a listing closes within N days, and how long it stays open?
- Which features drive the prediction most?
- **Does the priority queue beat how a career changer searches today** (newest first, title keyword search)? Measured on the test month: of each career changer's top 10, how many are still open long enough to apply, realistically reachable, and in the role they actually moved into (from `ML_Fit`).
- Does adding career-change accessibility and fit change a career changer's top 10, compared with timing factors alone?
- How many of a career changer's top 10 listings are still open when they apply, and realistically reachable?
- **Ghost listings in training:** does the timing model predict the test month better when trained without `likely_ghost` listings (EDA_SDA_2 3.1d)? A ghost's days open isn't a real hiring window. Keep whichever wins; ghosts stay in the data and stay hidden in the app either way.
- Is the model fast enough to rank listings live in the app?

**Leakage rules** (`STANDARDS.md`):
- Listing features use only what's known when a listing first appears. Company hiring volume and reposts count only listings first seen **before** it; `last_seen_at`, `status` and days open are never features.
- **Date picker:** a model that scores a chosen date is trained only on what was known by then; listings still open that day are censored at that day. The main model is evaluated on the test months, and earlier dates need a model retrained as of that date.
- **Role model:** only `feature` columns from the Profiles modeling table. Compass and path counts used as features are rebuilt inside each training fold, so a test person never counts toward their own answer.
- **Claims:** the resumes show only people who made a switch, so role-model results describe where switchers went, never how likely a switch is to work.

**Data rules:** aggregates only. Clear outputs before committing.

### Roadmap and methods

| Section | Course standard (Intro2ML, SupervisedLearning, NumericalMethods, FinalProject, MLinBusiness) | Taken further |
|:--|:--|:--|
| 1. Setup | Load the modeling-ready table | The features are whatever columns `EDA_SDA_4` kept; none is named here |
| 2. Targets | Classification: closes within N days. Regression: days open | **Survival** target (days open with censoring) |
| 3. Preprocessing | Drop identifier columns; one-hot encoding with aligned columns; OrdinalEncoder for trees; **StandardScaler fit on training data only** | `Pipeline` + `ColumnTransformer`; rare levels grouped |
| 4. Split | Train / validation / test **once**, `random_state=12345`, test used once at the end | Split **by time** (train April–July, validate August, test September), folds **grouped by company**; fit scores come from `ML_Fit`, trained on its own person-grouped folds |
| 5. Baselines | **DummyClassifier / DummyRegressor**, majority class, random | Newest-first ranking; title keyword search (how a career changer searches today) |
| 6. Class imbalance | `class_weight='balanced'`, upsampling (training only) | Decision-threshold tuning |
| 7. Models | **LogisticRegression, DecisionTree, RandomForest, LightGBM, CatBoost, XGBoost**, GradientBoosting, each tuned with loops or **GridSearchCV + cross-validation**; **`%%time`** for speed | **Cox regression** (with proportional-hazards checks), accelerated failure time models, survival forests, competing-risks models; **learning-to-rank** (LightGBM LambdaRank), which optimizes the order of an applicant's list directly |
| 8. Evaluation | ROC-AUC, F1, precision–recall and average precision, accuracy, **confusion matrix** (interpreted correctly), classification report; RMSE / MAE / R²; training vs. validation to check overfitting | **C-index** and time-dependent Brier score for survival; ranking metrics **precision@k** and **NDCG**; calibration; **conformal prediction** for apply-by ranges with guaranteed coverage |
| 8d. The priority queue | | **Combine** `ML_Fit`'s fit score with this model's chance still open and apply-by date, career-change accessibility, pay and the TimeSeries features in one score: a formula with weights that start from the evidence (EDA_SDA_4's adjusted model) and are tuned on the validation month, with user sliders in the app. Only features that survived EDA_SDA_4 go in |
| 9. Interpretation | **Feature importance** from the final model | **SHAP** values and partial dependence. **Per-listing breakdown for the app's detail view:** each factor's 0–100 rating and its exact contribution to the priority score (rating × weight, adding up to the total), plus each listing's top SHAP features for chance still open, grouped under plain-language names; check that SHAP values add up to each prediction, and save the breakdown with the scores |
| 10. Business translation | Top-N selection by prediction; **bootstrap** confidence intervals and risk (MLinBusiness) | "Of a career changer's top 10, how many are still open and realistically reachable", with company-clustered intervals |
| 11. Sensitivity | | Do conclusions hold under our choices (2-day live tolerance, scope boundaries, CET `posted`)? |
| 12. Solution report | Steps performed, difficulties and how they were solved, the final model and its score | |

## 1. Setup

In [ ]:
# 1a Libraries and the shared helpers (src/eda_helpers.py: paths, survival curves, tables, charts)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, YELLOW, INK, INK_2, MUTED, CLEAN_DIR,
    as_list, bar_h, finish, km_curve, km_median, km_survival, pct, show, surv_at, use_chart_style,
)
use_chart_style()

In [ ]:
# 1b ML_Rank scaffolding (src/rank_model.py: targets, split, baselines, models, metrics, the priority formula)
from src import rank_model as rm

## 1c. Hypotheses and decision rule (written before any modeling)

The test month (September) is used once, at the end. **One rule decides H1 to H5 and H7:** the model beats its baseline, and the company-clustered 95% bootstrap interval for the difference (model minus baseline; whole companies resampled together) excludes zero (`rm.clustered_bootstrap_diff` for per-listing or per-person values, `rm.clustered_bootstrap_auc_diff` for ROC-AUC). No fixed cut-off numbers are set; the size of the difference is reported either way. The top-10 overlap test (H6) is secondary.

| # | Hypothesis | Compared with | Supported when |
|:--|:--|:--|:--|
| H1 | Features known at first sighting predict whether a listing closes within N days better than the prior | `DummyClassifier` (prior, AUC 0.5); ROC-AUC difference, company-clustered (AUC, not Brier: the balanced class weights leave the probabilities uncalibrated) | Difference positive, interval excludes 0 |
| H2 | The best model beats the logistic regression | Same, logistic regression as the baseline | Same (otherwise keep the simpler model) |
| H3 | The survival model orders days open better than chance | C-index against 0.5 (the chance baseline), company-clustered bootstrap | Interval for C-index minus 0.5 excludes 0 |
| H4 | The priority queue beats newest first | Share of each career changer's top 10 that is relevant (below), test month | Difference positive, interval excludes 0 |
| H5 | The queue beats title keyword search | Same | Same |
| H6 (secondary) | Adding accessibility and fit changes the top 10 versus timing alone | Overlap of the two top 10s (`rm.top_k_overlap`) | Overlap below 80% |
| H7 | Training without `likely_ghost` listings predicts the test month better | Same rule, trained with ghosts as the baseline | Same |

**N, the closing window.** One constant, `rm.CLOSING_DAYS`, drives the main target ("closes within N days"), "open long enough" (still open N days after the person first sees the listing) and every hypothesis. It is 14 for now (TimeSeries: 37-45% close within 14 days); 7, 14 and 30 days are repeated as a sensitivity check in section 11 (`rm.SENSITIVITY_DAYS`). Change it in `src/rank_model.py` only.

**Backtest relevance (H4, H5).** For each career changer, a listing in their top 10 is relevant if it is (1) still open long enough and not a likely ghost, (2) reachable, and (3) in the role that person actually moved into (`rm.precision_per_person`). H4 and H5 compare values per person, so the bootstrap resamples whole people.

## 2. Targets

Closes within N days (unknown, not "stayed open", when still open and seen for fewer than N days), days open (closed listings only), and the survival target (days open with censoring, so still-open listings are kept).

In [ ]:
# 2a Load the modeling-ready table: needed columns only
# EDA_SDA_4 saves modeling_ready.parquet: identifiers and outcome columns plus only the features its section 10 kept (by
# EDA_SDA_2's screening rule), one row per `id`. The features are read from the table, so they follow EDA_SDA_4's decisions.
import pyarrow.parquet as pq
MODELING_TABLE = CLEAN_DIR / "modeling_ready.parquet"
NOT_FEATURES = {"id", "company_id", "title", "first_seen", "last_seen", "month", "days_open", "live", "closed"}  # ids, the split and the targets
FEATURES = [c for c in pq.read_schema(MODELING_TABLE).names if c not in NOT_FEATURES]
listings = pd.read_parquet(MODELING_TABLE, columns=["id", "company_id", "title", "first_seen", "last_seen", *FEATURES])
CATEGORICAL = [c for c in FEATURES if listings[c].dtype == object or str(listings[c].dtype) in ("category", "string")]
NUMERIC = ["company_prior_listings"] + [c for c in FEATURES if c not in CATEGORICAL]  # company_prior_listings is computed in 3a
for c in NUMERIC[1:]:  # booleans and nullable numbers as plain floats; blank stays NaN (the median imputer and its was-blank flag handle it)
    listings[c] = pd.to_numeric(listings[c].astype("Float64"), errors="coerce").to_numpy(dtype=float, na_value=np.nan)
listings[CATEGORICAL] = listings[CATEGORICAL].astype(object)  # the imputer needs plain strings, not a category type
print(f"{len(listings):,} listings; {len(NUMERIC) - 1} numeric and {len(CATEGORICAL)} categorical features from EDA_SDA_4")


In [ ]:
# 2b Targets: the main window is rm.CLOSING_DAYS (one constant, see 1c)
N_DAYS = rm.CLOSING_DAYS
target = rm.survival_target(listings)
listings["closes_within_n"] = rm.closes_within(target, N_DAYS)
listings["days_open"] = rm.days_open(target)
print(f"Still open (censored): {1 - target['event'].mean():.1%}")
print(f"Closes within {N_DAYS} days: {listings['closes_within_n'].mean():.1%} of the listings with a known answer")

**What the data shows**
- (fill in after the run)

## 3. Preprocessing and leakage checks

Features come only from what is known at first sighting. Company counts use earlier listings only (`prior_company_counts`); `has_desc` and the text flags are checked below (EDA_SDA_2 3.9) before use. Preprocessing sits inside a `Pipeline` + `ColumnTransformer`, fit on training data only.

In [ ]:
# 3a Leakage audit: company volume from earlier listings only; text flags unknown (not False) without a description
listings["company_prior_listings"] = rm.prior_company_counts(listings)
# FLAGS = ["urgency", "entry_wording", "changer_wording", "degree_required", "equivalent_experience"]
# print(rm.known_at_first_sight(features, FLAGS))  # expect 0 for every flag

**What the data shows**
- (fill in after the run)

### 3b Date picker check: July 1 vs. September 1, 2026

`src/as_of.py` returns the listings live on a date (first seen on or before it, last seen on or after it) with features from earlier listings only, and the training rows as known that day: a listing last seen more than `LIVE_TOLERANCE_DAYS` (2) before the date is finished; the rest are censored at the date. Counts only; no rows are shown. EDA_SDA_1 8b shows dates before July are skewed young, so the demo starts with later dates.

In [ ]:
# 3b Counts for the two dates (live listings, median age in days, companies, training rows and how many are censored)
from src.as_of import NEEDED, count_check  # imported here only: the date picker is used for this check
from src.eda_helpers import PART2_TABLE

as_of_listings = pd.read_parquet(PART2_TABLE, columns=NEEDED)
print(f"{len(as_of_listings):,} listings in the scope table")
count_check(as_of_listings, ("2026-07-01", "2026-09-01")).round(1)

**What the data shows**
- (fill in after the run)

Decision to record: which dates the app's picker offers (test-period dates first; earlier dates need a model retrained as of that date).

### 3c Preprocessing pipeline

`ColumnTransformer`: median for blank numbers (with a was-blank flag), "unknown" for blank categories, one-hot with levels under 50 listings grouped as "infrequent" and unseen levels allowed. Scaling sits inside the logistic regression only. The transformer is fit inside each training fold by the `Pipeline`, never on all rows.

In [ ]:
# 3c The preprocessor, shared by every model
PREP = rm.make_preprocessor(NUMERIC, CATEGORICAL, min_frequency=50)
PREP

**What the data shows**
- (fill in after the run)

## 4. Split

Train April–July (plus the data's first two days), validate August, test September. Folds grouped by company. The test month is touched once, in section 8.

In [ ]:
# 4a Time split and company-grouped folds
listings["split"] = rm.time_split(listings["first_seen"])
print(listings["split"].value_counts().to_string())
train = listings[listings["split"] == "train"].reset_index(drop=True)
folds = rm.company_folds(train["company_id"], n_splits=5)

**What the data shows**
- (fill in after the run)

In [ ]:
# 4b Rows with a known answer (censored-before-N rows are unknown, not "stayed open"), and X / y
FEATURES = NUMERIC + CATEGORICAL
listings["company_prior_listings"] = rm.prior_company_counts(listings)
known = listings[listings["closes_within_n"].notna()]
train = known[known["split"] == "train"].reset_index(drop=True)
valid = known[known["split"] == "validation"].reset_index(drop=True)
folds = rm.company_folds(train["company_id"], n_splits=5)
X_train, y_train = train[FEATURES], train["closes_within_n"].astype(int)
X_valid, y_valid = valid[FEATURES], valid["closes_within_n"].astype(int)
print(f"train {len(train):,} | validation {len(valid):,} | closes within {N_DAYS} days: {y_train.mean():.1%} / {y_valid.mean():.1%}")

**What the data shows**
- (fill in after the run)

## 5. Baselines

`DummyClassifier`, newest first, and title keyword search (the career changer's keywords come from `ML_Fit`'s role queries).

In [ ]:
# 5a Baseline scores on the validation month
from sklearn.metrics import roc_auc_score
valid_scores = {"newest first": rm.newest_first_score(valid)}
# valid_scores["title keywords"] = rm.title_keyword_score(valid, KEYWORDS)  # KEYWORDS: from ML_Fit
dummy = rm.dummy_baseline().fit(X_train, y_train)
prior_p = dummy.predict_proba(X_valid)[:, 1]
print(f"prior (DummyClassifier): AUC {roc_auc_score(y_valid, prior_p):.3f} (0.5 by construction)")
print(f"newest first as a score for 'closes within {N_DAYS} days': AUC {roc_auc_score(y_valid, valid_scores['newest first']):.3f}")

**What the data shows**
- (fill in after the run)

## 6. Class imbalance

Three ways to handle the imbalance between closes-within-N and stays-open, compared on the validation month with the logistic regression: no handling, `class_weight="balanced"` (what `rm.build_models()` sets), and upsampling the minority class (training rows only). Then the decision threshold is tuned on the validation month and applied unchanged to the test month.

In [ ]:
# 6a Imbalance handling: AUC and F1 at 0.5 on validation (training rows only are reweighted or upsampled)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def logit(**kw):
    return rm.with_preprocessing(PREP, make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=rm.SEED, **kw)))

rows = []
for name, model, (Xf, yf) in [("none", logit(), (X_train, y_train)),
                              ("class_weight balanced", logit(class_weight="balanced"), (X_train, y_train)),
                              ("upsampled", logit(), rm.upsample_minority(X_train, y_train))]:
    p = model.fit(Xf, yf).predict_proba(X_valid)[:, 1]
    rows.append({"handling": name, "AUC": roc_auc_score(y_valid, p), "F1 at 0.5": f1_score(y_valid, p >= 0.5, zero_division=0)})
pd.DataFrame(rows).set_index("handling").round(3)

**What the data shows**
- (fill in after the run)

In [ ]:
# 6b Decision threshold: chosen on validation (best F1), then fixed for the test month
bal = logit(class_weight="balanced").fit(X_train, y_train)
p_valid = bal.predict_proba(X_valid)[:, 1]
THRESHOLD, f1_best = rm.best_threshold(y_valid, p_valid, metric="f1")
print(f"threshold {THRESHOLD:.2f}: F1 {f1_best:.3f} (at 0.5: {f1_score(y_valid, p_valid >= 0.5, zero_division=0):.3f})")

**What the data shows**
- (fill in after the run)

## 7. Models

Logistic regression, decision tree, random forest, LightGBM, CatBoost, XGBoost (`rm.build_models()`), each with GridSearchCV over the company-grouped folds and `%%time`; Cox and survival models (`rm.cox_model()`, proportional-hazards check, company-clustered errors); LightGBM LambdaRank (`rm.lambdarank_model()`).

In [ ]:
# 7a Cross-validated ROC-AUC per model (company-grouped folds), against the dummy
models = {"dummy": dummy, **rm.build_models()}
cv_scores = {}
for name, m in models.items():
    pipe = rm.with_preprocessing(PREP, m)
    s = rm.cv_auc(pipe, X_train, y_train, folds)
    cv_scores[name] = s
    print(f"{name}: {s.mean():.3f} ± {s.std():.3f}")

**What the data shows**
- (fill in after the run)

### 7b Grid search

`GridSearchCV` over the company-grouped folds, one grid per model (`rm.PARAM_GRIDS`), timed with `%%time`. The settings chosen here are fixed before the validation month is scored.

In [ ]:
%%time
# 7b Grid search per model (ROC-AUC, company-grouped folds); best settings and CV mean ± std
searches = {}
for name, m in rm.build_models().items():
    gs = rm.grid_search(rm.with_preprocessing(PREP, m), rm.PARAM_GRIDS[name], X_train, y_train, folds)
    searches[name] = gs
    i = gs.best_index_
    print(f"{name}: {gs.best_score_:.3f} ± {gs.cv_results_['std_test_score'][i]:.3f}  {gs.best_params_}")

**What the data shows**
- (fill in after the run)

### 7c Cox proportional hazards (company-clustered errors) with the proportional-hazards check

The survival target keeps still-open listings (censored). A small p-value in the check means a feature's effect changes over a listing's life, so its hazard ratio is not one constant number. These estimates feed `START_WEIGHTS` for the queue.

In [ ]:
# 7c Cox on the training months: numeric features only here (add dummies for categories as EDA_SDA_4's adjusted model does)
surv_train = target.loc[listings.index[listings["split"] == "train"]].copy()
surv_train = surv_train.join(listings[["company_id"] + NUMERIC])
surv_train[NUMERIC] = surv_train[NUMERIC].fillna(surv_train[NUMERIC].median())
cox = rm.fit_cox(surv_train, NUMERIC)
display(cox.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(3))
display(rm.cox_ph_check(cox, surv_train, NUMERIC).round(3))

**What the data shows**
- (fill in after the run)

### 7d Accelerated failure time models

Weibull, log-normal and log-logistic AFT models say how much a feature stretches or shrinks the time open (a ratio on days), without the proportional-hazards assumption. Compared on C-index and AIC.

In [ ]:
# 7d AFT models (durations floored at half a day)
aft_df = rm.aft_ready(surv_train[NUMERIC + ["duration", "event"]])
rows = []
for kind in ["weibull", "lognormal", "loglogistic"]:
    m = rm.aft_model(kind).fit(aft_df, "duration", "event")
    rows.append({"model": kind, "AIC": m.AIC_, "C-index": m.concordance_index_})
pd.DataFrame(rows).set_index("model").round(3)

**What the data shows**
- (fill in after the run)

### 7e Random survival forest

Non-linear effects and interactions without assuming a hazard shape. Fit on a random sample of training listings to keep memory in check; scored by C-index on the validation month.

In [ ]:
%%time
# 7e Random survival forest (scikit-survival)
SURV_ROWS = 100_000
st = surv_train.sample(min(SURV_ROWS, len(surv_train)), random_state=rm.SEED)
rsf = rm.survival_forest().fit(st[NUMERIC], rm.survival_y(st["duration"], st["event"]))
surv_valid = target.loc[listings.index[listings["split"] == "validation"]].join(listings[NUMERIC])
surv_valid[NUMERIC] = surv_valid[NUMERIC].fillna(surv_train[NUMERIC].median())
risk = rsf.predict(surv_valid[NUMERIC])
print(f"validation C-index: {rm.c_index(surv_valid['duration'], surv_valid['event'], -risk):.3f}")

**What the data shows**
- (fill in after the run)

### 7f LambdaRank

Learning to rank optimizes the order of a list directly. A "query" is one list that gets ranked; relevance is graded (0 = closed within N days, 1 = stayed open). Until `ML_Fit`'s career-changer lists are wired in, the query here is the day a listing first appeared (rank that day's new listings), as a stand-in; `rm.backtest_relevance` defines the final relevance.

In [ ]:
%%time
# 7f LambdaRank, trained on the training months, scored with NDCG@10 on the validation month
import numpy as np
def day_queries(df):
    return df["first_seen"].dt.strftime("%Y-%m-%d").values

Xt = PREP.fit(X_train).transform(X_train)
ranker = rm.train_lambdarank(pd.DataFrame(Xt), 1 - y_train.values, day_queries(train))
scores = ranker.predict(PREP.transform(X_valid))
ndcgs = [rm.ndcg_at_k(1 - y_valid.values[idx], scores[idx], 10)
         for idx in pd.Series(range(len(valid))).groupby(day_queries(valid)).apply(lambda s: s.values) if len(idx) >= 10]
print(f"mean NDCG@10 over {len(ndcgs)} validation days: {np.mean(ndcgs):.3f}")

**What the data shows**
- (fill in after the run)

## 8. Evaluation

ROC-AUC, F1, precision-recall, confusion matrix and classification report; C-index for survival; precision@k and NDCG for rankings. Each hypothesis is judged by the 1c rule (beats the baseline, company-clustered interval excludes zero). The test month is used once, with the settings and threshold fixed on validation.

In [ ]:
# 8a Validation: the chosen model against the logistic regression and the prior (H1, H2); test is run once in 8c
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
best_name = max(searches, key=lambda k: searches[k].best_score_)
best = searches[best_name].best_estimator_
p_best = best.predict_proba(X_valid)[:, 1]
print(f"best model: {best_name} | AUC {roc_auc_score(y_valid, p_best):.3f} | average precision {average_precision_score(y_valid, p_best):.3f}")
print(confusion_matrix(y_valid, p_best >= THRESHOLD))
print(classification_report(y_valid, p_best >= THRESHOLD, zero_division=0))
h1 = rm.clustered_bootstrap_auc_diff(y_valid, p_best, prior_p, valid["company_id"])  # AUC: fair to balanced-weight models
h2 = rm.clustered_bootstrap_auc_diff(y_valid, p_best, p_valid, valid["company_id"])
print("H1 vs prior:", {k: round(v, 4) if isinstance(v, float) else v for k, v in h1.items()})
print("H2 vs logistic:", {k: round(v, 4) if isinstance(v, float) else v for k, v in h2.items()})

**What the data shows**
- (fill in after the run)

In [ ]:
# 8b Survival ordering (H3): C-index on validation, company-clustered bootstrap for C minus 0.5
cs = rm.c_index(surv_valid["duration"], surv_valid["event"], -risk)
comp = listings.loc[surv_valid.index, "company_id"].values
rng = np.random.default_rng(rm.SEED)
codes = pd.factorize(comp)[0]
members = [np.where(codes == k)[0] for k in range(codes.max() + 1)]
boot = []
for _ in range(100):  # small number: the C-index is slow on large tables; raise on a capable machine
    pick = np.concatenate([members[k] for k in rng.integers(0, len(members), size=len(members))])
    boot.append(rm.c_index(surv_valid["duration"].values[pick], surv_valid["event"].values[pick], -risk[pick]))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"C-index {cs:.3f}, company-clustered 95% interval {lo:.3f} to {hi:.3f}; beats chance (0.5): {lo > 0.5}")

**What the data shows**
- (fill in after the run)

### 8c Test month (run once)

Set `RUN_TEST = True` only after every choice above is final. Same metrics as 8a with the validation threshold, no further tuning.

In [ ]:
# 8c Test month, once
RUN_TEST = False
if RUN_TEST:
    test = known[known["split"] == "test"].reset_index(drop=True)
    p_test = best.predict_proba(test[FEATURES])[:, 1]
    y_test = test["closes_within_n"].astype(int)
    print(f"test AUC {roc_auc_score(y_test, p_test):.3f}, F1 at the validation threshold {f1_score(y_test, p_test >= THRESHOLD):.3f}")

**What the data shows**
- (fill in after the run)

### 8d The priority queue (H4, H5, H6)

Fit score, chance still open, accessibility and pay in one score (`rm.queue_top_k`, the same sum as `rm.priority_breakdown`): each factor rated 0–100, times its weight. EDA_SDA_4's Cox hazard ratios describe closing speed, so they do not set these four weights (they enter through the listing model's chance still open).

**The backtest.** One month's listings that a career changer sees (the 8 fit roles; not a hidden occupation; not manager level or above) with a known N-day answer, ranked for each career changer from `ML_Fit` (`fit_scores.parquet`):
- **Fit:** the person's `ML_Fit` probability for the listing's role, rated so their best role is 100.
- **Chance still open:** 1 minus the chosen listing model's chance of closing within N days (8a).
- **Accessibility:** how many of the career-change signs a listing shows (entry seniority, 2 years or fewer asked, entry wording, career-changer wording; EDA_SDA_2 3.11).
- **Pay:** hourly pay, rated within its role; missing pay is neutral (50).
- **Relevant:** stayed open N days, not a likely ghost, and in the role the person actually moved into.

**Leakage:** the weights are tuned on August with the **development people**, whose fit scores are out-of-fold; September is scored once, with the **test people**, whose fit scores come from a model trained without them. Relevance uses outcomes (closing, ghost), never as an input.

**Baselines:** newest first (the same list for everyone), and a title search for the most common role among the development people (ML_Fit's baseline) sorted newest first. The search is a perfect one (every listing in that role and nothing else), so it flatters the baseline.

In [ ]:
# 8d-a The backtest pool and people; the four weights tuned on August with the development people
from src.eda_helpers import PART2_TABLE
fit_table = pd.read_parquet(CLEAN_DIR / "fit_scores.parquet")
FIT_ROLES = [c.removeprefix("fit: ") for c in fit_table.columns if c.startswith("fit: ")]
pool_info = pd.read_parquet(PART2_TABLE, columns=["id", "level", "hidden_occupation", "likely_ghost", "salary_hourly_usd", *rm.ACCESS_SIGNS]).set_index("id")
MANAGER_LEVELS = ["manager", "leadership"]  # hidden for career changers (EDA_SDA_2 3.11)


def as_bool(s):
    return s.astype("boolean").fillna(False).to_numpy(dtype=bool)


def month_pool(split):
    """One month's listings a career changer sees, with a known N-day answer: factors, first_seen and `ok`
    (stayed open N days and not a likely ghost; an outcome, used only to judge the lists)."""
    m = known[(known["split"] == split) & known["role_bucket"].astype(str).isin(FIT_ROLES)]
    info = pool_info.reindex(m["id"].values)
    keep = ~as_bool(info["hidden_occupation"]) & ~info["level"].astype(str).isin(MANAGER_LEVELS).to_numpy()
    m, info = m[keep].reset_index(drop=True), info[keep]
    return pd.DataFrame({"role_bucket": m["role_bucket"].astype(str).values,
                         "p_still_open": 1 - best.predict_proba(m[FEATURES])[:, 1],
                         "accessibility": rm.accessibility_count(info).values,
                         "pay": info["salary_hourly_usd"].values,
                         "first_seen": m["first_seen"].values,
                         "ok": (m["closes_within_n"].to_numpy() == 0) & ~as_bool(info["likely_ghost"])})


def people(split):
    """Fit ratings (people x roles), the role each person moved into, and their ids."""
    g = fit_table[fit_table["split"] == split].reset_index(drop=True)
    fit = g[[f"fit: {r}" for r in FIT_ROLES]].set_axis(FIT_ROLES, axis=1)
    return rm.fit_ratings(fit), g["target_bucket"].to_numpy(dtype=object), g["person_id"].to_numpy()


def lists_for(pool, fit_rating, weights):
    """Each method's top 10 for every person (pool positions)."""
    ratings, roles, n = rm.listing_ratings(pool), pool["role_bucket"], len(fit_rating)
    return {"queue (tuned weights)": rm.queue_top_k(fit_rating, ratings, roles, weights),
            "queue (start weights)": rm.queue_top_k(fit_rating, ratings, roles, rm.START_WEIGHTS),
            "chance still open only": rm.queue_top_k(fit_rating, ratings, roles, {"still_open": 1.0}),
            "newest first": rm.newest_top_k(pool["first_seen"], n),
            f"title search: {MOST_COMMON}": rm.newest_top_k(pool["first_seen"], n, roles=roles, search_role=MOST_COMMON)}


def summarize(tops, pool, targets):
    """Per method: share of each top 10 that is relevant, open long enough, and in the role moved into (means over people)."""
    ok, roles = pool["ok"].to_numpy(), pool["role_bucket"].to_numpy(dtype=object)
    return pd.DataFrame({name: {"relevant (precision@10)": rm.precision_per_person(t, ok, roles, targets).mean(),
                                "open long enough, not a ghost": ok[t].mean(),
                                "in the role moved into": (roles[t] == targets[:, None]).mean()}
                         for name, t in tops.items()}).T


aug = month_pool("validation")
fit_dev, moved_dev, _ = people("development")
MOST_COMMON = pd.Series(moved_dev).mode()[0]
WEIGHTS, p_tuned, tried = rm.tune_queue_weights(fit_dev, rm.listing_ratings(aug), aug["role_bucket"], aug["ok"], moved_dev)
print(f"August pool: {len(aug):,} listings, {aug['ok'].mean():.1%} open long enough and not a ghost | {len(fit_dev):,} development people")
print(f"tuned weights {WEIGHTS} (precision@10 {p_tuned:.3f}; start weights {rm.START_WEIGHTS}); title search baseline: {MOST_COMMON}")
summarize(lists_for(aug, fit_dev, WEIGHTS), aug, moved_dev).round(3)

**What the data shows**
- (fill in after the run; the tuned row is optimistic on August, where its weights were chosen)

### 8e The priority queue on the test month (run once)

The weights from 8d, the test people, September's listings. H4 and H5: the tuned queue against newest first and the title search, per person, with a 95% bootstrap interval over people (`rm.clustered_bootstrap_diff`, one cluster per person). H6: how much of the queue's top 10 is also in the top 10 by chance still open alone (`rm.mean_overlap`; below 0.8 = they differ). Runs with `RUN_TEST` (8c).

In [ ]:
# 8e Test month, once: H4, H5, H6
if RUN_TEST:
    sep = month_pool("test")
    fit_test, moved_test, test_ids = people("test")
    tops = lists_for(sep, fit_test, WEIGHTS)
    display(summarize(tops, sep, moved_test).round(3))
    ok, roles = sep["ok"].to_numpy(), sep["role_bucket"].to_numpy(dtype=object)
    per_person = {name: rm.precision_per_person(t, ok, roles, moved_test) for name, t in tops.items()}
    queue = per_person["queue (tuned weights)"]
    for h, base in [("H4", "newest first"), ("H5", f"title search: {MOST_COMMON}")]:
        r = rm.clustered_bootstrap_diff(queue, per_person[base], test_ids)
        print(f"{h} queue minus {base}: {r['difference']:+.3f} [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}], supported: {r['supported']}")
    overlap = rm.mean_overlap(tops["queue (tuned weights)"], tops["chance still open only"])
    print(f"H6 overlap with chance still open only: {overlap:.1%}; the lists differ (below 80%): {overlap < 0.8}")
    print(f"September pool: {len(sep):,} listings | {len(fit_test):,} test people")

**What the data shows**
- (fill in after the run)

## 9. Interpretation

SHAP and partial dependence; the per-listing breakdown for the detail view (each factor's rating and share of the score from `priority_breakdown`, top SHAP features behind chance still open under plain-language names; check SHAP adds up to each prediction); saved with the scores.

_To do._

## 10. Business translation

Top-10 results with company-clustered bootstrap intervals.

_To do._

## 11. Sensitivity

The 7, 14 and 30 day windows (`rm.SENSITIVITY_DAYS`) for the closing target, the live tolerance (`rm.LIVE_TOLERANCE_DAYS`), scope boundaries, CET `posted`, with and without ghost listings (H7).

In [ ]:
# 11a Does the result hold at 7, 14 and 30 days? Share closing, and cross-validated AUC of the tuned model, per window
rows = []
windows = rm.sensitivity_windows(target)
for n in rm.SENSITIVITY_DAYS:
    y_n = windows[f"closes_within_{n}"].reindex(listings.index)
    tr = listings[(listings["split"] == "train") & y_n.notna()].reset_index(drop=True)
    f = rm.company_folds(tr["company_id"], n_splits=5)
    pipe = rm.with_preprocessing(PREP, best.named_steps["model"])
    s = rm.cv_auc(pipe, tr[FEATURES], y_n[(listings["split"] == "train") & y_n.notna()].astype(int).values, f)
    rows.append({"window (days)": n, "share closing": y_n.mean(), "known answers": int(y_n.notna().sum()), "CV AUC mean": s.mean(), "std": s.std()})
pd.DataFrame(rows).set_index("window (days)").round(3)

**What the data shows**
- (fill in after the run)

_Still to do: live tolerance, scope boundaries, CET `posted`, and H7 (ghosts)._

## 12. Solution report

Steps performed, difficulties and how they were solved, the final model and its score.

_To do._